# Editing analysis

Compilation of editing results and generation of MLE for each sensor.

In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [9]:
sample_key = pd.read_excel('../../screening_data/sample_key.xlsx')

#exclude SY-5609 since this is same as CBE subpool 1
sample_key = sample_key[sample_key['Screen_data_location']!='SY_5609'].reset_index(drop=True)

In [13]:
#define the editing files that match up with the samples (2 for each)
file1 = []
file2 = []

fp = '../../screening_data'

for i, val in sample_key.iterrows():
    fn = val['File_Name']
    loc = val['Screen_data_location']

    fp2 = f'{fp}/{loc}_screen_data/editing'

    files = os.listdir(fp2)
    files_relevant = sorted([i for i in files if fn in i])

    file1.append(f'{fp2}/{files_relevant[0]}')
    file2.append(f'{fp2}/{files_relevant[1]}')

sample_key['file1'] = file1
sample_key['file2'] = file2

sample_key

,File_Name,Sample,Editor,Screen_data_location,Subpool,file1,file2
0,D24-257001,Plasmid,PLASMID,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
...,...,...,...,...,...,...,...
99,D25-188027,ABE_CDK12-IN-2_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
100,D25-188028,ABE_HQ461_REP1,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
101,D25-188029,ABE_HQ461_REP2,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
102,D25-188030,ABE_HQ461_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...


In [22]:
#and create groups for MLE calculation
g = []
for i, val in sample_key.iterrows():
    name = val['Sample']
    if name not in ['Plasmid', 'PLASMID_LIBRARY']:
        g.append(name[:-5])
    elif name in ['Plasmid', 'PLASMID_LIBRARY']:
        g.append('Plasmid')

sample_key['group'] = g
sample_key

,File_Name,Sample,Editor,Screen_data_location,Subpool,file1,file2,group
0,D24-257001,Plasmid,PLASMID,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,Plasmid
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
...,...,...,...,...,...,...,...,...
99,D25-188027,ABE_CDK12-IN-2_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_CDK12-IN-2
100,D25-188028,ABE_HQ461_REP1,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461
101,D25-188029,ABE_HQ461_REP2,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461
102,D25-188030,ABE_HQ461_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461


In [54]:
def HGVSp_combiner(subset):

    file_name_list = []
    for i, val in subset.iterrows():
        f1 = val['file1']
        f2 = val['file2']
        file_name_list.append(f1)
        file_name_list.append(f2)

    df_holder = []
    for k in file_name_list:
        d = pd.read_csv(k)
        df_holder.append(d)

    #and the combining
    new_out = pd.concat(df_holder).groupby(['Edit', 'HGVSp', 'Num_edits', 'DNA Change', 'Canonical_edit', 'Canonical_window', 'gRNA_id']).sum().reset_index().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    hgvsp_only = new_out[['gRNA_id', 'HGVSp', '#Reads']].groupby(['gRNA_id', 'HGVSp'], as_index=False).sum().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    #and calculate percentages efficiently
    new_out['%Reads'] = (new_out['#Reads'] / new_out.groupby('gRNA_id')['#Reads'].transform('sum')) * 100

    hgvsp_only['%Reads'] = (hgvsp_only['#Reads'] / hgvsp_only.groupby('gRNA_id')['#Reads'].transform('sum')) * 100


    return new_out, hgvsp_only

In [53]:
5964/10266

0.5809468147282291

In [69]:
#need to group together subpool because of redo of T0_REP3 for CBE in Subpool1 (sequenced with ABE)
s = [['ABE_subpool_1', 'CBE_subpool_1'], ['CDK12_13'], ['KB_compound_mut']]
s_key = ['subpool1', 'CDK12_13', 'KB_compound_mut']
s_key2 = ['subpool1', 'subpool2', 'subpool1']

for idx, i in enumerate(s):
    subset = sample_key[sample_key['Screen_data_location'].isin(i)]
    true_name = s_key[idx]
    true_name2 = s_key2[idx]

    CBE_portion = subset[subset['Editor']=='CBE']
    ABE_portion = subset[subset['Editor']=='ABE']
    Plasmid = subset[subset['Editor']=='PLASMID']

    CBE_groups = np.unique(CBE_portion['group'])
    ABE_groups = np.unique(ABE_portion['group'])
    Plasmid_groups = np.unique(Plasmid['group'])

    for k in CBE_groups:
        subset = CBE_portion[CBE_portion['group']==k].reset_index(drop=True)
        new_out, hgvsp_only = HGVSp_combiner(subset)

        fp = f'../../screening_data/04_editing/CBE_{true_name}'
        new_out.to_csv(f'{fp}/full/{k}_full.csv', index=False)
        hgvsp_only.to_csv(f'{fp}/HGVSp_only/{k}_HGVSp.csv', index=False)

    for j in ABE_groups:
        subset = ABE_portion[ABE_portion['group']==j].reset_index(drop=True)
        new_out, hgvsp_only = HGVSp_combiner(subset)
        fp = f'../../screening_data/04_editing/ABE_{true_name}'
        new_out.to_csv(f'{fp}/full/{j}_full.csv', index=False)
        hgvsp_only.to_csv(f'{fp}/HGVSp_only/{j}_HGVSp.csv', index=False)

    for b in Plasmid_groups:
        subset = Plasmid[Plasmid['group']==b].reset_index(drop=True)
        new_out, hgvsp_only = HGVSp_combiner(subset)

        fp = f'../../screening_data/04_editing/Plasmid'
        new_out.to_csv(f'{fp}/{true_name2}/full/{b}_{true_name2}_full.csv', index=False)
        hgvsp_only.to_csv(f'{fp}/{true_name2}/HGVSp_only/{b}_{true_name2}_HGVSp.csv', index=False)



# MLE generation

Combining all samples to generate maximum likelihood estimate. These will be the editing files used for downstream analysis.

In [107]:
def MLE_generator(folders):

    fp = '../../screening_data/04_editing/'

    full = []
    hg = []
    for folder in folders:
        fp_full = f'{fp}/{folder}/full'
        fp_hgvsp = f'{fp}/{folder}/HGVSp_only'


        full_list = os.listdir(fp_full)
        hg_list = os.listdir(fp_hgvsp)

        full_list = [i for i in full_list if '.csv' in i]
        hg_list = [i for i in hg_list if '.csv' in i]

        print(full_list)

        for k in full_list:
            j = pd.read_csv(f'{fp_full}/{k}')
            full.append(j)

        for k in hg_list:
            j = pd.read_csv(f'{fp_hgvsp}/{k}')
            hg.append(j)

        #and then generate the MLE
        new_out = pd.concat(full).groupby(['Edit', 'HGVSp', 'Num_edits', 'DNA Change', 'Canonical_edit', 'Canonical_window', 'gRNA_id']).sum().reset_index().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)
        hgvsp_only = new_out[['gRNA_id', 'HGVSp', '#Reads']].groupby(['gRNA_id', 'HGVSp'], as_index=False).sum().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

        new_out['%Reads'] = (new_out['#Reads'] / new_out.groupby('gRNA_id')['#Reads'].transform('sum')) * 100
        hgvsp_only['%Reads'] = (hgvsp_only['#Reads'] / hgvsp_only.groupby('gRNA_id')['#Reads'].transform('sum')) * 100

        #and then compress and save
        compression_options = dict(method='zip', archive_name=f'{fp}/{folder}_full.csv')
        new_out.to_csv(f'{fp}/{folder}_full.zip', compression=compression_options, index=False)

        compression_options2 = dict(method='zip', archive_name=f'{fp}/{folder}_HGVSp.csv')
        hgvsp_only.to_csv(f'{fp}/{folder}_HGVSp.zip', compression=compression_options2, index=False)
        
        


In [108]:
#ignore plasmid
folders = ['CBE_CDK12_13',
 'ABE_CDK12_13',
 'CBE_KB_compound_mut',
 'CBE_subpool1',
 'ABE_subpool1']

MLE_generator(folders)

['CBE_CDK12-IN-2_full.csv', 'CBE_T0_full.csv', 'CBE_HQ461_full.csv', 'CBE_DMSO_full.csv', 'CBE_BSJ-4-116_full.csv']
['ABE_HQ461_full.csv', 'ABE_DMSO_full.csv', 'ABE_CDK12-IN-2_full.csv', 'ABE_BSJ-4-116_full.csv', 'ABE_T0_full.csv']
['DMSO_full.csv', 'KB_4000_full.csv', 'KB_2000_full.csv', 'T0_full.csv']
['KI-CDK9d-32_1000nM_full.csv', 'DMSO_full.csv', 'KI-CDK9d-32_100nM_full.csv', 'KI-CDK9d-32N_1250nM_full.csv', 'SEL120_4000nM_full.csv', 'KB-0742_1500nM_full.csv', 'KI-CDK9d-32N_5000nM_full.csv', 'Senexin B_15000nM_full.csv', 'Senexin B_2000nM_full.csv', 'T0_full.csv']
['KI-CDK9d-32_1000nM_full.csv', 'DMSO_full.csv', 'KI-CDK9d-32_100nM_full.csv', 'KI-CDK9d-32N_1250nM_full.csv', 'SEL120_4000nM_full.csv', 'KB-0742_1500nM_full.csv', 'KI-CDK9d-32N_5000nM_full.csv', 'Senexin B_15000nM_full.csv', 'Senexin B_2000nM_full.csv', 'T0_full.csv']


In [109]:
pd.read_csv('../../screening_data/04_editing/ABE_CDK12_13_HGVSp.zip')

,gRNA_id,HGVSp,#Reads,%Reads
0,gRNA_CDK12_targ_1740,WT,13939,29.756212
1,gRNA_CDK12_targ_1740,M1V,11051,23.591068
2,gRNA_CDK12_targ_1740,P2L,10885,23.236701
3,gRNA_CDK12_targ_1740,P2F,2057,4.391171
4,gRNA_CDK12_targ_1740,M1V_N3D,1730,3.693109
...,...,...,...,...
758270,gRNA_CDK13_targ_6272,R1508T_G1509Y_L1510*_P1511A,1,0.003207
758271,gRNA_CDK13_targ_6272,R1508T_G1509Y_L1510*_Y1512*,1,0.003207
758272,gRNA_CDK13_targ_6272,R1508T_Y1512H,1,0.003207
758273,gRNA_CDK13_targ_6272,Y1512C,1,0.003207
